# Advanced Python Modules for Bioinformatics
## BIOINF 575 - Complete Guide to Module Development and Debugging

**Objectives:** Master Python module creation, CLI tools, and debugging for production bioinformatics workflows.

**Prerequisites:** Basic Python (variables, functions, classes, file I/O)

**What you'll build:** A complete bioinformatics package with CLI interface, error handling, and debugging tools.

This notebook covers: module structure, import patterns, CLI parsing (sys/argparse), debugging (pdb/Jupyter), and production deployment.[web:42][web:45]

## 1. Module Fundamentals

### What is a Python Module?
- A `.py` file containing Python code (functions, classes, variables)
- Can be imported into other Python files/notebooks
- Promotes code reuse and organization

### Why Modules Matter in Bioinformatics
- **Reusability:** Write FASTA parsers once, use everywhere
- **Maintainability:** Fix bugs in one place, affects all projects
- **Collaboration:** Team members can import shared utilities
- **Testing:** Unit test individual components independently

**Example Use Cases:**
- `sequence_utils.py`: GC content, reverse complement, translation
- `file_parsers.py`: FASTA, GFF, VCF readers
- `statistical_tests.py`: t-tests, ANOVA for differential expression

## 2. Creating Your First Bioinformatics Module

Let's build a `bioinformatics_utils` module with common sequence analysis functions.

In [ ]:
%%writefile bioinformatics_utils.py
"""
Bioinformatics Utilities Module
Common functions for sequence analysis and file handling

Author: BIOINF 575
Version: 1.0
"""

import os
from typing import List, Tuple, Optional
import logging

# Set up logging for debugging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def gc_content(sequence: str) -> float:
    """
    Calculate GC content percentage of a DNA sequence.
    
    Args:
        sequence: DNA sequence (string)
    
    Returns:
        GC content as percentage (0-100)
    
    Raises:
        ValueError: If sequence contains invalid characters
    """
    sequence = sequence.upper().replace(' ', '')
    
    # Validate DNA sequence
    valid_bases = set('ATCG')
    if not all(base in valid_bases for base in sequence):
        invalid = set(sequence) - valid_bases
        raise ValueError(f"Invalid bases found: {invalid}")
    
    g_count = sequence.count('G')
    c_count = sequence.count('C')
    total = len(sequence)
    
    if total == 0:
        return 0.0
    
    return (g_count + c_count) / total * 100

def reverse_complement(sequence: str) -> str:
    """
    Calculate reverse complement of DNA sequence.
    
    Args:
        sequence: DNA sequence
    
    Returns:
        Reverse complement sequence
    """
    complement_map = str.maketrans('ATCG', 'TAGC')
    return sequence.upper().translate(complement_map)[::-1]

def parse_fasta_file(filepath: str) -> List[Tuple[str, str]]:
    """
    Parse FASTA file and return list of (header, sequence) tuples.
    
    Args:
        filepath: Path to FASTA file
    
    Returns:
        List of (header, sequence) tuples
    
    Raises:
        FileNotFoundError: If file doesn't exist
        ValueError: If file is not valid FASTA format
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"FASTA file not found: {filepath}")
    
    sequences = []
    current_header = None
    current_seq = []
    
    with open(filepath, 'r') as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if line.startswith('>'):
                # Save previous sequence
                if current_header is not None:
                    seq = ''.join(current_seq)
                    sequences.append((current_header, seq))
                
                current_header = line[1:]  # Remove >
                current_seq = []
            else:
                # Sequence line
                if not line:  # Skip empty lines
                    continue
                current_seq.append(line)
        
        # Don't forget the last sequence
        if current_header is not None:
            seq = ''.join(current_seq)
            sequences.append((current_header, seq))
    
    if not sequences:
        raise ValueError("No valid sequences found in FASTA file")
    
    logging.info(f"Parsed {len(sequences)} sequences from {filepath}")
    return sequences

def find_motifs(sequence: str, motif: str, min_occurrences: int = 1) -> List[Tuple[int, int]]:
    """
    Find all occurrences of a motif in a sequence.
    
    Args:
        sequence: DNA sequence to search
        motif: Motif pattern to find
        min_occurrences: Minimum number of occurrences required
    
    Returns:
        List of (start, end) positions where motif is found
    """
    positions = []
    start = 0
    motif = motif.upper()
    
    while True:
        pos = sequence.upper().find(motif, start)
        if pos == -1:
            break
        positions.append((pos, pos + len(motif)))
        start = pos + 1  # Overlapping search
    
    if len(positions) < min_occurrences:
        logging.warning(f"Found only {len(positions)} occurrences, need at least {min_occurrences}")
    
    return positions

class SequenceAnalyzer:
    """Class for comprehensive sequence analysis."""
    
    def __init__(self, sequence: str):
        self.sequence = sequence.upper()
        self.length = len(sequence)
        self.gc = gc_content(sequence)
    
    def analyze(self) -> dict:
        """Perform comprehensive analysis."""
        return {
            'length': self.length,
            'gc_content': self.gc,
            'at_content': 100 - self.gc,
            'reverse_complement': reverse_complement(self.sequence),
            'base_counts': self._count_bases()
        }
    
    def _count_bases(self) -> dict:
        """Count individual bases."""
        return {
            'A': self.sequence.count('A'),
            'T': self.sequence.count('T'),
            'G': self.sequence.count('G'),
            'C': self.sequence.count('C')
        }

# Module-level variables (use sparingly)
VERSION = "1.0.0"
SUPPORTED_FILE_FORMATS = ['fasta', 'fastq', 'genbank']

def main():
    """Module main function - runs when executed as script."""
    print(f"Bioinformatics Utils v{VERSION}")
    print(f"Supported formats: {', '.join(SUPPORTED_FILE_FORMATS)}")
    
if __name__ == "__main__":
    main()

### 🧪 Testing Your Module

**Import different ways:**

In [ ]:
# Method 1: Import entire module
import bioinformatics_utils as bio

print(f"Module version: {bio.VERSION}")
print(f"Supported formats: {bio.SUPPORTED_FILE_FORMATS}")

In [ ]:
# Method 2: Import specific functions
from bioinformatics_utils import gc_content, reverse_complement, parse_fasta_file

# Test functions
dna_seq = "ATGGCCATCG"
print(f"Original: {dna_seq}")
print(f"GC content: {gc_content(dna_seq):.1f}%")
print(f"Reverse complement: {reverse_complement(dna_seq)}")

In [ ]:
# Method 3: Import class
from bioinformatics_utils import SequenceAnalyzer

analyzer = SequenceAnalyzer("ATGGCCATCG")
analysis = analyzer.analyze()
print("Complete analysis:")
for key, value in analysis.items():
    print(f"  {key}: {value}")

### 🚀 Exercise 1 (10 min)

1. Add a function `translate_dna` to your module that translates DNA to protein
2. Handle stop codons and invalid codons
3. Test with: `ATGGCCATGSTOP` (should stop at STOP)
4. Add proper error handling for non-DNA input

```python
# Your functions/classes here

def main():
    # Demo/test code
    pass

if __name__ == "__main__":
    main()
```

In [ ]:
# Test the __name__ behavior
print(f"Running in notebook, __name__ = '{__name__}'")

# Now test the module's __name__
import bioinformatics_utils
print(f"Module __name__ = '{bioinformatics_utils.__name__}'")

# Run as script (in terminal)
!python bioinformatics_utils.py

**Bioinformatics Example:** 

Create a module that can either:
- Be imported for its functions (sequence analysis)
- Run as a script to process FASTA files from command line

## 4. Command Line Interfaces (CLI) - 3 Approaches

### 4.1 Basic: `sys.argv` (Simple Scripts)

In [ ]:
%%writefile simple_cli.py
#!/usr/bin/env python3
"""
Simple CLI using sys.argv
Usage: python simple_cli.py input.fasta output.txt
"""
import sys
from bioinformatics_utils import parse_fasta_file, gc_content

def main():
    # sys.argv is script name, [1:] are arguments
    if len(sys.argv) != 3:
        print(f"Usage: {sys.argv} <input.fasta> <output.txt>")
        sys.exit(1)
    
    input_file = sys.argv
    output_file = sys.argv
    
    print(f"Processing {input_file}...")
    
    try:
        sequences = parse_fasta_file(input_file)
        results = []
        
        for header, seq in sequences:
            gc = gc_content(seq)
            results.append(f">{header}\nGC: {gc:.2f}%\n{seq[:50]}...\n")
        
        with open(output_file, 'w') as f:
            f.write("".join(results))
        
        print(f"Results written to {output_file}")
    
    except Exception as e:
        print(f"Error: {e}")
        sys.exit(1)

if __name__ == "__main__":
    main()

**Test sys.argv:**

In [ ]:
# Create test FASTA file
%%writefile test_sequences.fasta
>TP53
ATGGAGCCGGCGCATGGC
>BRCA1
GCGCGCGCGCGCGCGCGC
# Done

In [ ]:
# Test the simple CLI
!python simple_cli.py test_sequences.fasta output.txt
!cat output.txt

### 4.2 Intermediate: `getopt` (Traditional Unix Style)

In [ ]:
%%writefile getopt_cli.py
#!/usr/bin/env python3
"""
CLI using getopt - Unix-style options
Usage: python getopt_cli.py -i input.fasta -o output.txt [-m motif] [-t threshold]
"""
import sys
import getopt
from bioinformatics_utils import parse_fasta_file, gc_content, find_motifs
import json

def main():
    try:
        # Define options: short (-i) and long (--input) forms
        opts, args = getopt.getopt(
            sys.argv[1:], 
            'i:o:m:t:',  # Short options (colon = requires argument)
            ['input=', 'output=', 'motif=', 'threshold=']
        )
    
        # Default values
        input_file = None
        output_file = None
        motif = "GC"
        gc_threshold = 50.0
        output_format = "txt"
    
        # Parse options
        for opt, arg in opts:
            if opt in ('-i', '--input'):
                input_file = arg
            elif opt in ('-o', '--output'):
                output_file = arg
            elif opt in ('-m', '--motif'):
                motif = arg
            elif opt in ('-t', '--threshold'):
                gc_threshold = float(arg)
            elif opt == '-f':
                output_format = arg
    
        # Validate required arguments
        if not input_file or not output_file:
            print("Error: -i/--input and -o/--output are required")
            print("Usage: python getopt_cli.py -i input.fasta -o output.txt [-m motif] [-t threshold]")
            sys.exit(1)
    
        print(f"Analyzing {input_file} with motif '{motif}' (GC > {gc_threshold}%)...")
    
        # Process file
        sequences = parse_fasta_file(input_file)
        results = []
    
        for header, seq in sequences:
            gc = gc_content(seq)
            positions = find_motifs(seq, motif)
            
            result = {
                'header': header,
                'gc_content': gc,
                'motif_hits': len(positions),
                'positions': positions,
                'above_threshold': gc > gc_threshold
            }
            results.append(result)
    
        # Output results
        if output_format == 'json':
            with open(output_file, 'w') as f:
                json.dump(results, f, indent=2)
        else:
            with open(output_file, 'w') as f:
                f.write(f"Sequence Analysis Results\n")
                f.write(f"{'='*50}\n\n")
                
                for result in results:
                    status = "PASS" if result['above_threshold'] else "FAIL"
                    f.write(f">{result['header']} [{status}]\n")
                    f.write(f"  GC: {result['gc_content']:.2f}%\n")
                    f.write(f"  Motif '{motif}' hits: {result['motif_hits']}\n")
                    if result['positions']:
                        f.write(f"  Positions: {result['positions']}\n")
                    f.write("\n")
    
        print(f"Analysis complete. Results saved to {output_file}")
        print(f"Found {len([r for r in results if r['above_threshold']])} sequences above GC threshold")

    except getopt.GetoptError as err:
        print(f"Error parsing arguments: {err}")
        print("Usage: python getopt_cli.py -i input.fasta -o output.txt [-m motif] [-t threshold]")
        sys.exit(2)
    except Exception as e:
        print(f"Processing error: {e}")
        sys.exit(1)

if __name__ == "__main__":
    main()

In [ ]:
# Test getopt CLI
!python getopt_cli.py -i test_sequences.fasta -o results_getopt.txt -m GC -t 60
!cat results_getopt.txt

In [ ]:
# Test JSON output and error handling
!python getopt_cli.py -i test_sequences.fasta -o results.json -f json
!python getopt_cli.py -h  # Should show error for missing required args

### 4.3 Advanced: `argparse` (Modern, User-Friendly)

`argparse` is the gold standard for Python CLI tools - automatic help, type checking, and better error messages.

In [ ]:
%%writefile argparse_cli.py
#!/usr/bin/env python3
"""
Professional Bioinformatics CLI Tool using argparse

Features:
- Automatic help generation
- Type checking and validation
- Subcommands for different analyses
- Progress bars and logging
- Configuration file support
"""
import argparse
import sys
import logging
from pathlib import Path
from typing import List, Dict
import json
from tqdm import tqdm  # Progress bars
from bioinformatics_utils import (
    parse_fasta_file, gc_content, reverse_complement, find_motifs, SequenceAnalyzer
)

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('bioinformatics.log'),
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)

class SmartFormatter(argparse.HelpFormatter):
    """Custom formatter for better help text."""
    def _split_lines(self, text, width):
        if text.startswith('R|'):
            return text[2:].splitlines()
        return argparse.HelpFormatter._split_lines(self, text, width)

def validate_fasta(parser, arg):
    """Type validator for FASTA files."""
    path = Path(arg)
    if not path.exists():
        parser.error(f"FASTA file does not exist: {arg}")
    if not path.suffix.lower() in ['.fasta', '.fa', '.fastq']:
        parser.error(f"File does not appear to be FASTA format: {arg}")
    return str(path.absolute())

def validate_positive_float(parser, arg):
    """Validator for positive float values."""
    try:
        value = float(arg)
        if value < 0:
            parser.error(f"Value must be positive: {arg}")
        return value
    except ValueError:
        parser.error(f"Invalid float value: {arg}")

def gc_analysis(args):
    """Subcommand: GC content analysis."""
    logger.info(f"Starting GC analysis on {args.input}")
    
    try:
        sequences = parse_fasta_file(args.input)
        results = []
        
        # Progress bar
        for header, seq in tqdm(sequences, desc="Analyzing sequences"):
            analyzer = SequenceAnalyzer(seq)
            analysis = analyzer.analyze()
            
            result = {
                'header': header,
                'length': analysis['length'],
                'gc_content': analysis['gc_content'],
                'at_content': analysis['at_content'],
                'base_counts': analysis['base_counts'],
                'above_threshold': analysis['gc_content'] > args.threshold,
                'reverse_complement_preview': analysis['reverse_complement'][:50] + "..."
            }
            results.append(result)
        
        # Filter results
        high_gc = [r for r in results if r['above_threshold']]
        low_gc = [r for r in results if not r['above_threshold']]
        
        # Summary statistics
        all_gc = [r['gc_content'] for r in results]
        summary = {
            'total_sequences': len(results),
            'high_gc_count': len(high_gc),
            'low_gc_count': len(low_gc),
            'mean_gc': sum(all_gc) / len(all_gc),
            'gc_range': (min(all_gc), max(all_gc)),
            'threshold': args.threshold
        }
        
        # Save results
        output_data = {
            'summary': summary,
            'sequences': results
        }
        
        with open(args.output, 'w') as f:
            json.dump(output_data, f, indent=2)
        
        logger.info(f"Analysis complete. {len(high_gc)}/{len(results)} sequences above threshold")
        logger.info(f"Results saved to {args.output}")
        
        # Print summary
        print(f"\n=== SUMMARY ===")
        print(f"Total sequences: {summary['total_sequences']}")
        print(f"High GC (>{args.threshold}%): {summary['high_gc_count']}")
        print(f"Mean GC content: {summary['mean_gc']:.2f}%")
        
        if args.verbose:
            print(f"\nHigh GC sequences:")
            for result in high_gc[:5]:  # Top 5
                print(f"  {result['header']}: {result['gc_content']:.2f}%")
    
    except Exception as e:
        logger.error(f"GC analysis failed: {e}")
        sys.exit(1)

def motif_analysis(args):
    """Subcommand: Motif searching."""
    logger.info(f"Searching for motif '{args.motif}' in {args.input}")
    
    try:
        sequences = parse_fasta_file(args.input)
        results = []
        
        for header, seq in tqdm(sequences, desc="Searching motifs"):
            positions = find_motifs(seq, args.motif, args.min_hits)
            
            result = {
                'header': header,
                'sequence_length': len(seq),
                'motif': args.motif,
                'hits': len(positions),
                'positions': positions,
                'meets_criteria': len(positions) >= args.min_hits
            }
            results.append(result)
        
        # Save results
        with open(args.output, 'w') as f:
            json.dump(results, f, indent=2)
        
        # Summary
        total_hits = sum(r['hits'] for r in results)
        qualifying = sum(1 for r in results if r['meets_criteria'])
        
        print(f"\n=== MOTIF ANALYSIS SUMMARY ===")
        print(f"Motif: {args.motif}")
        print(f"Total sequences: {len(results)}")
        print(f"Sequences with ≥{args.min_hits} hits: {qualifying}")
        print(f"Total motif occurrences: {total_hits}")
        print(f"Results saved to {args.output}")
        
    except Exception as e:
        logger.error(f"Motif analysis failed: {e}")
        sys.exit(1)

def create_parser() -> argparse.ArgumentParser:
    """Create the main argument parser with subcommands."""
    parser = argparse.ArgumentParser(
        description="Professional Bioinformatics Analysis Toolkit",
        formatter_class=SmartFormatter,
        epilog="R|Examples:\n"
                "  python argparse_cli.py gc -i sequences.fasta -o results.json -t 60 --verbose\n"
                "  python argparse_cli.py motif -i genes.fasta -o motifs.json -m TATA -n 3\n"
                "  python argparse_cli.py --version"
    )
    
    # Global arguments
    parser.add_argument('--version', action='version', version='Bioinformatics CLI 2.0')
    parser.add_argument('-v', '--verbose', action='store_true', help='Enable verbose output')
    parser.add_argument('--log-level', choices=['DEBUG', 'INFO', 'WARNING', 'ERROR'],
                       default='INFO', help='Logging level')
    
    # Subparsers for different commands
    subparsers = parser.add_subparsers(dest='command', help='Analysis type',
                                     required=True)
    
    # GC Analysis subcommand
    gc_parser = subparsers.add_parser('gc', help='GC content analysis',
                                    description='Analyze GC content of sequences',
                                    formatter_class=SmartFormatter)
    gc_parser.add_argument('-i', '--input', required=True, type=validate_fasta,
                          help='Input FASTA file', metavar='FILE')
    gc_parser.add_argument('-o', '--output', required=True,
                          help='Output JSON file', metavar='FILE')
    gc_parser.add_argument('-t', '--threshold', type=validate_positive_float,
                          default=50.0, help='GC threshold (default: 50.0)')
    gc_parser.add_argument('--min-length', type=int, default=0,
                          help='Minimum sequence length to include')
    gc_parser.set_defaults(func=gc_analysis)
    
    # Motif Analysis subcommand
    motif_parser = subparsers.add_parser('motif', help='Motif searching',
                                       description='Search for motifs in sequences')
    motif_parser.add_argument('-i', '--input', required=True, type=validate_fasta,
                             help='Input FASTA file')
    motif_parser.add_argument('-o', '--output', required=True,
                             help='Output JSON file')
    motif_parser.add_argument('-m', '--motif', default='GC',
                             help='Motif to search for (default: GC)')
    motif_parser.add_argument('-n', '--min-hits', type=int, default=1,
                             help='Minimum number of motif hits required')
    motif_parser.add_argument('--case-sensitive', action='store_true',
                             help='Case sensitive motif matching')
    motif_parser.set_defaults(func=motif_analysis)
    
    return parser

def main():
    """Main entry point."""
    parser = create_parser()
    args = parser.parse_args()
    
    # Set logging level
    logging.getLogger().setLevel(getattr(logging, args.log_level))
    
    logger.info(f"Starting {args.command} analysis")
    
    # Run the appropriate function
    try:
        args.func(args)
    except KeyboardInterrupt:
        logger.info("Analysis interrupted by user")
        sys.exit(130)
    except Exception as e:
        logger.error(f"Unexpected error: {e}")
        sys.exit(1)

if __name__ == "__main__":
    main()

### 🧪 Test the Professional CLI

In [ ]:
# Install tqdm for progress bars (if needed)
# !pip install tqdm

# Test help - should show beautiful formatted help
!python argparse_cli.py -h

In [ ]:
# Test GC analysis subcommand
!python argparse_cli.py gc -i test_sequences.fasta -o gc_results.json -t 60 -v

In [ ]:
# Test motif analysis
!python argparse_cli.py motif -i test_sequences.fasta -o motif_results.json -m GC -n 2

In [ ]:
# Test error handling
!python argparse_cli.py gc -i nonexistent.fasta -o test.json

### 🚀 Exercise 2 (20 min) - Build Your Own CLI

Create `variant_analyzer.py` that:
1. Accepts VCF files as input
2. Has subcommands: `count`, `filter`, `annotate`
3. Uses argparse with proper validation
4. Includes progress bars and logging
5. Handles common bioinformatics errors

**Test cases:**
- `python variant_analyzer.py count -i variants.vcf`
- `python variant_analyzer.py filter -i variants.vcf -o filtered.vcf --min-depth 20`

## 5. Debugging Techniques

### 5.1 Print Debugging (Quick and Dirty)

In [ ]:
%%writefile buggy_sequence_analyzer.py
#!/usr/bin/env python3
"""
Example with common bioinformatics bugs + print debugging
"""
from bioinformatics_utils import parse_fasta_file, gc_content

def analyze_genome(input_file, output_file):
    print(f"DEBUG: Starting analysis of {input_file}")
    
    # Bug 1: No file existence check
    print(f"DEBUG: File exists? {os.path.exists(input_file)}")
    
    sequences = parse_fasta_file(input_file)
    print(f"DEBUG: Parsed {len(sequences)} sequences")
    
    results = []
    for i, (header, seq) in enumerate(sequences):
        print(f"DEBUG: Processing sequence {i+1}: {header[:20]}... (length: {len(seq)})")
        
        # Bug 2: No length validation
        if len(seq) == 0:
            print(f"WARNING: Empty sequence found: {header}")
            continue
        
        # Bug 3: GC calculation error handling
        try:
            gc = gc_content(seq)
            print(f"DEBUG: GC content for {header}: {gc:.2f}%")
        except ValueError as e:
            print(f"ERROR: {e} for sequence {header}")
            gc = 0.0
        
        results.append({'header': header, 'gc': gc, 'length': len(seq)})
    
    # Bug 4: Division by zero in statistics
    total_length = sum(r['length'] for r in results)
    avg_gc = sum(r['gc'] for r in results) / len(results) if results else 0
    
    print(f"DEBUG: Total length: {total_length}, Average GC: {avg_gc:.2f}%")
    
    # Save results
    import json
    with open(output_file, 'w') as f:
        json.dump({
            'summary': {'total_sequences': len(results), 'total_length': total_length, 'avg_gc': avg_gc},
            'sequences': results
        }, f, indent=2)
    
    print(f"Analysis complete. Results saved to {output_file}")

if __name__ == "__main__":
    import sys
    if len(sys.argv) != 3:
        print(f"Usage: {sys.argv} <input.fasta> <output.json>")
        sys.exit(1)
    
    analyze_genome(sys.argv, sys.argv)

In [ ]:
# Test buggy code with print debugging
!python buggy_sequence_analyzer.py test_sequences.fasta debug_output.json

### 5.2 Professional Debugging with `pdb`

`pdb` is Python's built-in debugger - set breakpoints, step through code, inspect variables.

In [ ]:
%%writefile pdb_debug_example.py
#!/usr/bin/env python3
"""
Example of using pdb for debugging bioinformatics code
"""
import pdb  # Python Debugger
from bioinformatics_utils import parse_fasta_file, gc_content

def problematic_analysis(input_file):
    print("Starting problematic analysis...")
    
    # Set breakpoint - execution will pause here
    pdb.set_trace()
    
    sequences = parse_fasta_file(input_file)
    print(f"Parsed {len(sequences)} sequences")
    
    results = []
    for header, seq in sequences[:3]:  # First 3 only for demo
        # Another breakpoint
        print(f"Processing {header}")
        pdb.set_trace()
        
        # Simulate bug: wrong GC calculation
        buggy_gc = gc_content(seq.lower())  # Should be upper!
        results.append({'header': header, 'buggy_gc': buggy_gc})
    
    # Final breakpoint
    pdb.set_trace()
    return results

if __name__ == "__main__":
    import sys
    if len(sys.argv) != 2:
        print(f"Usage: {sys.argv} <input.fasta>")
        sys.exit(1)
    
    results = problematic_analysis(sys.argv)
    print("Results:", results)

**PDB Commands (when paused):**

| Command | Description |
|---------|-------------|
| `n` or `next` | Execute next line (step over) |
| `s` or `step` | Step into function calls |
| `c` or `continue` | Continue until next breakpoint |
| `p variable` | Print value of variable |
| `l` or `list` | Show source code around current line |
| `w` or `where` | Show call stack |
| `q` or `quit` | Exit debugger |
| `h` or `help` | Show all commands |

In [ ]:
# Run with PDB - will pause at first breakpoint
# When it pauses, try: n (next), p sequences (print variable), c (continue)
!python pdb_debug_example.py test_sequences.fasta

### 5.3 Debugging in Jupyter Lab

Jupyter Lab has excellent debugging support:

In [ ]:
# Example: Debug this cell in Jupyter Lab
def buggy_function(sequence):
    # Set a breakpoint by clicking the line number in Jupyter Lab
    # Or use: import pdb; pdb.set_trace()
    length = len(sequence)
    gc = gc_content(sequence.lower())  # BUG: should be upper()
    return length, gc

# Run and debug
test_seq = "atggcc"  # lowercase
result = buggy_function(test_seq)
print(f"Length: {result}, GC: {result:.1f}%")

```python
# After an exception, run:
%debug
```

### 5.4 Advanced Debugging: `ipdb` and `pdbpp`

Enhanced debuggers with better features:

In [ ]:
# Install enhanced debuggers
# !pip install ipdb pdbpp

%%writefile advanced_debug.py
#!/usr/bin/env python3
"""
Example using ipdb and pdb++ for better debugging experience
"""
import ipdb  # Better than pdb for interactive debugging
from bioinformatics_utils import gc_content

def complex_analysis(sequences):
    """Complex function with multiple potential failure points."""
    results = []
    
    for i, seq_info in enumerate(sequences):
        header, seq = seq_info
        
        # Enhanced breakpoint with context
        ipdb.set_trace(context=5)  # Show 5 lines of context
        
        try:
            # Simulate complex processing
            gc = gc_content(seq)
            rev_comp = reverse_complement(seq)
            
            # Check for suspicious values
            if gc > 100 or gc < 0:
                ipdb.set_trace()  # Conditional breakpoint
            
            results.append({
                'index': i,
                'header': header,
                'gc_content': gc,
                'reverse_complement_length': len(rev_comp)
            })
            
        except Exception as e:
            print(f"Error processing {header}: {e}")
            # Continue debugging even after errors
            ipdb.set_trace()
    
    return results

if __name__ == "__main__":
    from bioinformatics_utils import parse_fasta_file
    sequences = parse_fasta_file("test_sequences.fasta")
    results = complex_analysis(sequences)
    print(f"Processed {len(results)} sequences successfully")

**Enhanced Debugger Features:**
- `ipdb`: Works in Jupyter, better syntax highlighting
- `pdbpp`: Syntax-colored, tab completion, better stack traces
- `context=N`: Show N lines of source code around breakpoint
- Watch expressions: `watch len(seq)` to monitor variables

## 6. Essential Bioinformatics Libraries

### 6.1 Biopython - The Gold Standard

In [ ]:
# Install Biopython (if needed)
# !pip install biopython

from Bio import SeqIO, Entrez, AlignIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import Bio

print(f"Biopython version: {Bio.__version__}")

# Example: Working with sequences
dna = Seq("ATGGCCATGGCGCG")
print(f"DNA: {dna}")
print(f"Length: {len(dna)}")
print(f"GC content: {dna.gc_fraction() * 100:.1f}%")
print(f"Protein: {dna.translate()}")

# Create sequence record
record = SeqRecord(dna, id="TEST", description="Test sequence")
print(f"Record: {record.format('fasta')}")

### 6.2 Working with NCBI via Biopython

In [ ]:
# NCBI access example (set your email - required by NCBI)
Entrez.email = "your.email@university.edu"  # Replace with your email!

def fetch_sequence(accession):
    """Fetch sequence from NCBI."""
    try:
        handle = Entrez.efetch(db="nucleotide", id=accession, rettype="fasta", retmode="text")
        record = SeqIO.read(handle, "fasta")
        handle.close()
        return record
    except Exception as e:
        print(f"Error fetching {accession}: {e}")
        return None

# Fetch human TP53 gene (example accession)
tp53 = fetch_sequence("NM_000546")  # Human TP53 mRNA
if tp53:
    print(f"Fetched {tp53.id}")
    print(f"Length: {len(tp53.seq)}")
    print(f"First 50 bp: {tp53.seq[:50]}")
    print(f"Translated: {tp53.seq[:150].translate()}")

### 6.3 Pandas for Tabular Bioinformatics Data

In [ ]:
import pandas as pd

# Example: Process gene expression data
data = {
    'gene': ['TP53', 'BRCA1', 'EGFR', 'KRAS', 'MYC'],
    'expression_log2': [12.5, 8.2, 15.1, 9.8, 11.3],
    'chromosome': ['17', '17', '7', '12', '8'],
    'gc_content': [52.3, 48.7, 55.1, 49.2, 53.8]
}

df = pd.DataFrame(data)
print("Gene Expression Data:")
print(df)

# Common bioinformatics operations
print("\nHighly expressed genes (>12):")
high_expr = df[df['expression_log2'] > 12]
print(high_expr)

# Statistical analysis
print(f"\nMean expression: {df['expression_log2'].mean():.2f}")
print(f"Correlation with GC: {df['expression_log2'].corr(df['gc_content']):.3f}")

# Group by chromosome
print("\nExpression by chromosome:")
print(df.groupby('chromosome')['expression_log2'].mean())

### 6.4 Matplotlib/Seaborn for Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Generate synthetic gene expression data
genes = [f'Gene_{i}' for i in range(1, 101)]
expression = np.random.normal(10, 2, 100)  # log2 expression values
gc_contents = np.random.uniform(30, 70, 100)

# Create DataFrame
df_plot = pd.DataFrame({
    'Gene': genes,
    'Expression': expression,
    'GC_Content': gc_contents
})

# Create publication-quality plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Histogram of expression
ax1.hist(df_plot['Expression'], bins=20, alpha=0.7, color='skyblue', edgecolor='black')
ax1.set_xlabel('Log2 Expression')
ax1.set_ylabel('Frequency')
ax1.set_title('Gene Expression Distribution')
ax1.axvline(df_plot['Expression'].mean(), color='red', linestyle='--', label=f'Mean: {df_plot["Expression"].mean():.2f}')
ax1.legend()

# Scatter plot: Expression vs GC content
sns.scatterplot(data=df_plot, x='GC_Content', y='Expression', ax=ax2, alpha=0.6)
ax2.set_xlabel('GC Content (%)')
ax2.set_ylabel('Log2 Expression')
ax2.set_title('Expression vs GC Content')

# Add trend line
z = np.polyfit(df_plot['GC_Content'], df_plot['Expression'], 1)
p = np.poly1d(z)
ax2.plot(df_plot['GC_Content'], p(df_plot['GC_Content']), "r--", alpha=0.8, linewidth=1)

plt.tight_layout()
plt.savefig('gene_expression_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Publication-quality figure saved as 'gene_expression_analysis.png'")
print(f"Correlation: {df_plot['Expression'].corr(df_plot['GC_Content']):.3f}")

## 7. Creating a Complete Bioinformatics Package

Organize modules into a proper Python package structure.

In [ ]:
# Create package structure
import os

package_structure = """
bioinformatics_package/
├── setup.py                 # Installation script
├── README.md               # Documentation
├── requirements.txt        # Dependencies
├── bioinformatics/         # Main package
│   ├── __init__.py         # Makes directory a package
│   ├── sequences.py        # Sequence utilities
│   ├── parsers.py          # File parsers
│   ├── analysis.py         # Analysis functions
│   └── cli.py              # Command line interface
├── tests/                  # Unit tests
│   ├── __init__.py
│   ├── test_sequences.py
│   └── test_parsers.py
└── docs/                   # Documentation
    ├── conf.py
    └── index.rst
"""

print("Recommended Package Structure:")
print(package_structure)

In [ ]:
%%writefile bioinformatics_package/setup.py
from setuptools import setup, find_packages

with open("README.md", "r") as fh:
    long_description = fh.read()

setup(
    name="bioinformatics-package",
    version="0.1.0",
    author="BIOINF 575 Students",
    author_email="student@university.edu",
    description="A bioinformatics analysis package",
    long_description=long_description,
    long_description_content_type="text/markdown",
    url="https://github.com/bioinf575/package",
    packages=find_packages(),
    classifiers=[
        "Programming Language :: Python :: 3",
        "License :: OSI Approved :: MIT License",
        "Operating System :: OS Independent",
    ],
    python_requires='>=3.7',
    install_requires=[
        'biopython>=1.78',
        'pandas>=1.3',
        'matplotlib>=3.5',
        'tqdm',
        'numpy',
    ],
    entry_points={
        'console_scripts': [
            'bio-analyze=Bioinformatics.cli:main',
        ],
    },
    include_package_data=True,
    package_data={'': ['*.txt', '*.md']},
)

In [ ]:
bash
pip install -e .
# or
pip install git+https://github.com/yourusername/bioinformatics-package.git
```

## Usage

### Command Line Interface

```bash
bio-analyze gc --input sequences.fasta --output results.json --threshold 60
bio-analyze motif --input genes.fasta --motif TATA --min-hits 2
```

### Python API

```python
from Bioinformatics.sequences import gc_content, reverse_complement
from Bioinformatics.parsers import parse_fasta_file

sequences = parse_fasta_file('input.fasta')
for header, seq in sequences:
    print(f"{header}: GC={gc_content(seq):.2f}%")
```

## Features

- FASTA/FASTQ parsing
- GC content and motif analysis
- NCBI database integration
- Publication-quality plotting
- Robust error handling and logging

## Development

```bash
# Install development dependencies
pip install -r requirements-dev.txt

# Run tests
pytest tests/

# Build documentation
cd docs && make html
```

## License

MIT License - see LICENSE file for details.

In [ ]:
%%writefile bioinformatics_package/requirements.txt
biopython>=1.78
pandas>=1.3.0
matplotlib>=3.5.0
seaborn>=0.11.0
numpy>=1.21.0
tqdm>=4.62.0
scipy>=1.7.0

In [ ]:
%%writefile bioinformatics_package/bioinformatics/__init__.py
"""
Bioinformatics Package - Main __init__ file
"""

from .sequences import gc_content, reverse_complement, find_motifs
from .parsers import parse_fasta_file
from . import analysis

__version__ = "0.1.0"
__all__ = [
    'gc_content',
    'reverse_complement',
    'find_motifs',
    'parse_fasta_file',
    'analysis'
]

# Import CLI main function
from .cli import main as cli_main

def run_cli():
    """Run the command line interface."""
    cli_main()

### 🧪 Testing Package Installation

In [ ]:
# Install the package in development mode
# !pip install -e bioinformatics_package/

# Test import
try:
    from bioinformatics import gc_content, __version__
    print(f"Package imported successfully! Version: {__version__}")
    
    # Test function
    test_gc = gc_content("GCGCGCGC")
    print(f"Test GC content: {test_gc:.1f}%")
    
    # Test CLI entry point (if installed)
    # !bio-analyze --help
    
except ImportError as e:
    print(f"Import error (normal if not installed): {e}")
    print("To test full installation, run: pip install -e bioinformatics_package/")

Project structure:
my_bio_project/
├── src/                    # Source code
│   └── bioinformatics/
├── tests/                  # Unit tests
├── docs/                   # Documentation
├── data/                   # Sample data
│   ├── raw/
│   └── processed/
├── notebooks/              # Analysis notebooks
├── scripts/                # Standalone scripts
├── environment.yml         # Conda environment
├── setup.py                # Package setup
├── README.md              # Project documentation
└── .gitignore             # Git ignore file
```

### 🔧 Development Workflow
1. **Prototype** in Jupyter notebooks
2. **Refactor** into modules/functions
3. **Test** with unit tests (`pytest`)
4. **Document** with docstrings and README
5. **Package** with `setup.py`
6. **Version control** with Git
7. **Deploy** to PyPI or GitHub

### 🐛 Debugging Checklist
- [ ] Use print statements for quick checks
- [ ] Set breakpoints with `pdb.set_trace()` or Jupyter Lab
- [ ] Check file paths and existence
- [ ] Validate input data types and ranges
- [ ] Handle edge cases (empty files, missing values)
- [ ] Test with small datasets before large runs
- [ ] Use logging instead of print for production code

### 📦 Essential Libraries
| Category | Libraries | Use Case |
|----------|-----------|----------|
| Core | `biopython`, `pandas`, `numpy` | Sequence analysis, data manipulation |
| Viz | `matplotlib`, `seaborn`, `plotly` | Publication-quality plots |
| Stats | `scipy`, `statsmodels` | Statistical analysis |
| CLI | `argparse`, `click`, `typer` | Command line interfaces |
| Testing | `pytest`, `hypothesis` | Unit testing and property-based testing |
| Docs | `sphinx`, `mkdocs` | Documentation generation |

### 🚀 Production Tips
- **Virtual environments:** Always use `venv` or `conda`
- **Type hints:** Use `typing` module for better IDE support
- **Documentation:** Write docstrings for all public functions
- **Version control:** Commit often with meaningful messages
- **Testing:** Aim for 80%+ code coverage
- **Licensing:** Choose appropriate license (MIT for open source)
- **Reproducibility:** Pin dependency versions in `requirements.txt`

## 🎯 Final Project: Build Your Bioinformatics Tool

**Create a complete bioinformatics package that:**

1. **Parses biological files** (FASTA, GFF, VCF, or BED)
2. **Performs analysis** (your choice: motif finding, variant annotation, expression analysis)
3. **Has CLI interface** with subcommands and proper argument parsing
4. **Includes debugging** with logging and error handling
5. **Has unit tests** covering main functionality
6. **Generates visualizations** (plots, heatmaps, etc.)
7. **Is installable** via `pip install -e .`

**Deliverables:**
- Complete package directory structure
- `setup.py` and `README.md`
- Example usage notebook
- Test data files
- Installation and usage instructions

**Example Tools to Build:**
- **Motif Scanner:** Find transcription factor binding sites
- **Variant Annotator:** Annotate VCF files with gene info
- **Expression Analyzer:** Differential expression from count tables
- **Genome Statistics:** GC content, gene density across chromosomes
- **Primer Designer:** Design PCR primers from sequences

**Success Criteria:**
- [ ] Package installs without errors
- [ ] CLI works with help text and validation
- [ ] All functions have docstrings
- [ ] Tests pass (`pytest tests/`)
- [ ] Handles common errors gracefully
- [ ] Generates meaningful output (files, plots)
- [ ] Includes example usage

**Timeline:** 2 weeks - plenty of time to build something substantial!

**Resources:**
- [Biopython Tutorial](https://biopython.org/wiki/Documentation)
- [Python Packaging Guide](https://packaging.python.org)
- [Click Documentation](https://click.palletsprojects.com) (alternative to argparse)
- [Pytest Documentation](https://docs.pytest.org)
- [Cookiecutter for packages](https://cookiecutter.readthedocs.io)